In [1]:
# Install the Hugging Face Datasets library used to load the source datasets.
!pip install -q datasets

In [2]:
# Load the iamtangsang Nepali-English translation dataset.
from datasets import load_dataset, concatenate_datasets

iamts = load_dataset("iamTangsang/Nepali-to-English-Translation-Dataset")
print(iamts)

README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 76.8MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.19MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/702697 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10866 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10867 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source', 'target'],
        num_rows: 702697
    })
    validation: Dataset({
        features: ['source', 'target'],
        num_rows: 10866
    })
    test: Dataset({
        features: ['source', 'target'],
        num_rows: 10867
    })
})


In [3]:
# Merge the train, validation, and test splits so they can be processed together.
iamtangsang = concatenate_datasets([
    iamts["train"],
    iamts["validation"],
    iamts["test"]
])

In [4]:
# Inspect the available columns, dataset size, and overall structure.
print(iamtangsang.column_names)
print(iamtangsang)
print(iamtangsang[0])

['source', 'target']
Dataset({
    features: ['source', 'target'],
    num_rows: 724430
})
{'source': '"कुनै पनि अन्य सरकारी एजेन्सीले यो जानकारी प्रयोग गर्न सक्दैन, केन्द्रीय सरकार अन्तर्गतका कसैले कुनै पनि हालतमा यो जानकारी पाउँदैनन् र राज्यका अधिकारीहरूमा पनि स्वास्थ्य अधिकारीहरूले मात्र यसलाई प्रयोग गर्न सक्दछन्," उनले भने।', 'target': '"No other government agency can use this information, no one in the commonwealth government at all, and in state authorities, only the health officer can use it.'}


In [5]:
# Load the HIMALAYA Nepali SFT dataset containing conversational examples.
himalaya = load_dataset("himalaya-ai/nepali-sft-dataset")
print(himalaya)

merged_dataset.jsonl: reconstructing file:   0%|          |  0.00B / 3.80GB            

merged_dataset.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1112863 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['conversations', 'source', 'id'],
        num_rows: 1112863
    })
})


In [6]:
print(himalaya["train"][0])

{'conversations': [{'from': 'system', 'value': 'You are a helpful AI assistant that answers in Nepali.'}, {'from': 'human', 'value': 'नेपालको इतिहासमा मध्यकालीन अवधिका राजनीतिक, आर्थिक, सामाजिक र सांस्कृतिक परिवर्तनहरूको बारेमा संक्षिप्त वर्णन गर्नुहोस्।'}, {'from': 'gpt', 'value': 'मध्यकालीन अवधिमा, नेपालमा राजनीतिक अस्थिरता देखियो र ससाना राज्यहरूको उदय र अवसान भयो। यस अवधिमा आर्थिक, सामाजिक र सांस्कृतिक अवस्थामा पनि परिवर्तन आयो। राजनीतिक रूपमा, नेपाल विभिन्न ससाना राज्यहरूमा विभाजित थियो, र यी राज्यहरूमा आफ्नो-आफ्नो शासकहरू थिए। यस्ता ससाना राज्यहरूको उदय र अवसानले गर्दा नेपालको राजनीतिक अवस्था अस्थिर थियो।\n\nआर्थिक रूपमा, मध्यकालीन नेपालमा कृषि, व्यापार र उद्योगमा परिवर्तन आयो। कृषि मुख्य आर्थिक गतिविधि थियो, र धेरैजसो मानिसहरू कृषिमा निर्भर थिए। व्यापार र उद्योग पनि विकसित भइरहेको थियो, तर यो सीमित थियो।\n\nसामाजिक रूपमा, मध्यकालीन नेपालमा विभिन्न जात र वर्गहरू थिए। समाजमा ऊँच-नीचको अवधारणा प्रचलित थियो, र उच्च वर्गका मानिसहरूको निम्न वर्गका मानिसहरूमाथि अधिकार थियो।\n\nसांस्कृत

#### Extracting translation pairs from Himalaya-sft dataset


In [7]:
# retrieve messages by their conversation role.

def get_message(example, role):
    for message in example["conversations"]:
        if message["from"] == role:
            return message["value"]
    return ""

def get_human(example):
    return get_message(example, "human")

def get_gpt(example):
    return get_message(example, "gpt")


In [8]:
# Identify HIMALAYA examples whose human message contains a translation instruction.

import re

translation_pattern = re.compile(
    r"^Translate the following (?:Nepali text to English|English text to Nepali)\s*:?\s*"
)

def has_translation_pattern(example):
    return bool(translation_pattern.search(get_human(example)))

translation_candidates = himalaya["train"].filter(
    has_translation_pattern,
    desc="Searching for translation samples",
)

print("Total translation candidates:", len(translation_candidates))


Searching for translation samples:   0%|          | 0/1112863 [00:00<?, ? examples/s]

Total translation candidates: 375491


In [9]:
# Patterns used to identify the translation direction from the instruction.
eng_to_nep_pattern = re.compile(
    r"translate\s+the\s+following\s+english\s+text\s+to\s+nepali",
    re.IGNORECASE,
)

nep_to_eng_pattern = re.compile(
    r"translate\s+the\s+following\s+nepali\s+text\s+to\s+english",
    re.IGNORECASE,
)

In [10]:

def classify_translation(example):
    human = get_human(example)

    if eng_to_nep_pattern.search(human):
        return "eng_to_nep"
    if nep_to_eng_pattern.search(human):
        return "nep_to_eng"
    return "unknown"

In [11]:
# Convert the selected HIMALAYA examples into a simple tabular structure.
import pandas as pd

results = []

for row in translation_candidates:
    results.append({
        "human": get_human(row),
        "gpt": get_gpt(row),
        "direction": classify_translation(row),
    })

translation_df = pd.DataFrame(results)

print("Direction counts:")
display(translation_df["direction"].value_counts().rename_axis("direction").to_frame("count"))

Direction counts:


,count
direction,
nep_to_eng,198159
eng_to_nep,177332


In [12]:
eng_to_nep = translation_df[translation_df["direction"] == "eng_to_nep"]
nep_to_eng = translation_df[translation_df["direction"] == "nep_to_eng"]

display(eng_to_nep.sample(min(10, len(eng_to_nep)), random_state=42))
display(nep_to_eng.sample(min(10, len(nep_to_eng)), random_state=42))

,human,gpt,direction
172569,Translate the following English text to Nepali...,ठेगाना पुस्तकको पहुँचका लागि प्रमाणीकरण आवश्यक...,eng_to_nep
2319,Translate the following English text to Nepali...,आर्थिक वर्ष २०२०।२१ देखि २०२६-२७ सम्मको ७ वर्ष...,eng_to_nep
109329,Translate the following English text to Nepali...,म्याकग्र हिललाई अपमान गरिएको थियो ।,eng_to_nep
2852,Translate the following English text to Nepali...,ख्याको हिसावले सहकारिताको विकास निकै अगाडि बढे...,eng_to_nep
153847,Translate the following English text to Nepali...,बाट जडान विच्छेद गर्दै,eng_to_nep
59139,Translate the following English text to Nepali...,(च) “पालनपोषण तथा हेरचाह” भन्नाले ज्येष्ठ नागर...,eng_to_nep
65929,Translate the following English text to Nepali...,(२) खेलकुदको विकास तथा प्रतियोगिता आयोजना गर्न...,eng_to_nep
126030,Translate the following English text to Nepali...,चेतावनीवाभ प्ले अडियो सर्भर का लागि तपाईँको गि...,eng_to_nep
115496,Translate the following English text to Nepali...,सेकेण्ड बाँणको रेखा चौडाइ,eng_to_nep
40272,Translate the following English text to Nepali...,मेरो ह्रदयमा विश्वास छ। मैले कुनै त्रुटी गरेको...,eng_to_nep


,human,gpt,direction
352650,Translate the following Nepali text to English...,Georgia Gulf rebuffed that offer in September ...,nep_to_eng
351941,Translate the following Nepali text to English...,"We work together, to meet the needs of our cli...",nep_to_eng
219814,Translate the following Nepali text to English...,The downgrading of debt issued by CS First Bos...,nep_to_eng
285647,Translate the following Nepali text to English...,We would like to think you are ok with this bu...,nep_to_eng
243515,Translate the following Nepali text to English...,"""How have you helped him who is without power!...",nep_to_eng
249015,Translate the following Nepali text to English...,Then our sons will be like well-nurtured plant...,nep_to_eng
273851,Translate the following Nepali text to English...,Thanks again for providing such wonderful soft...,nep_to_eng
223730,Translate the following Nepali text to English...,All shipments are insured and the buyer assume...,nep_to_eng
343900,Translate the following Nepali text to English...,She designed the uniforms for the 2012 Jamaica...,nep_to_eng
183312,Translate the following Nepali text to English...,- overcrowded living where one streptococci in...,nep_to_eng


Remove the translation instruction from the human column and keep only the actual text to be translated.

In [13]:
# Remove the instruction prefix and keep only the text that should be translated.
def extract_text(human_text):
    if pd.isna(human_text):
        return ""

    text = str(human_text).strip()

    # Remove translation instruction
    patterns = [
        r"^Translate the following Nepali text to English\s*:?\s*",
        r"^Translate the following English text to Nepali\s*:?\s*",
    ]

    for pattern in patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)

    return text.strip()


In [14]:
# Apply instruction removal to every Himalaya human message.
translation_df["human"] = translation_df["human"].apply(extract_text)

In [15]:
display(
    translation_df[
        ["human", "gpt", "direction"]
    ].sample(20, random_state=42)
)


,human,gpt,direction
163418,Recalculate Document,कागजात पुनगणना गर्नुहोस्,eng_to_nep
251321,कमल र नलिनीले वाल-सुलभ भावनाद्वारा प्रेरित आफ्...,Kamal and IFJ wall - accessible motivated by h...,nep_to_eng
214506,तपाईँले स्थितिपट्टीमा प्याडलक प्रतिमाबाट सधैँ ...,You can always see the security status of a pa...,nep_to_eng
68551,To give opinion and advice to the Executive Co...,(छ) प्राज्ञिक विषयमा कार्यकारी परिषद्लाई राय स...,eng_to_nep
98263,Name of the drug :–,१. औषधिको नामः–,eng_to_nep
343214,जंगल सकिनासाथ पूर्व र उत्तरको कोण लाग्नु भन्ने...,As per the direction of Vaidhraj to take angle...,nep_to_eng
146783,Rhythmbox manages all of your music in a centr...,"रिदमबक्सले एउटा केन्द्रिय सङ्गीत ""library"" मा ...",eng_to_nep
334959,हामी यस्तो समयमा छौं जब यो अत्यावश्यक छ कि हाम...,We are in a time where it is imperative that w...,nep_to_eng
113128,"However, the countryâ€™s basic structure if th...",तर पनि देशको आधारभूत अर्थव्यवस्था अझ निकै कमजो...,eng_to_nep
29892,But ten men were found among those who said to...,"तर ती मध्ये दशजना मानिसहरूले इश्माएललाई भने, “...",eng_to_nep


Standardize iamtangsang

In [16]:
# Convert iamtangsang to pandas and standardize its columns and metadata.
iam_df = iamtangsang.to_pandas()

iam_df = iam_df[["source", "target"]].copy()

iam_df["direction"] = "nep_to_eng"
iam_df["dataset"] = "iamTangsang"

Standardize himalaya ne-en

In [17]:
# Rename columns to the common schema.
himalaya_ne_en = translation_df[
    translation_df["direction"] == "nep_to_eng"][
    ["human", "gpt"]
].copy()
himalaya_ne_en = himalaya_ne_en.rename(
    columns={
        "human": "source",
        "gpt": "target"
    }
)
himalaya_ne_en["direction"] = "nep_to_eng"
himalaya_ne_en["dataset"] = "himalaya_nep_eng"


Standardize himalaya en-ne

In [18]:
himalaya_en_ne = translation_df[
    translation_df["direction"] == "eng_to_nep"][
    ["human", "gpt"]
].copy()
himalaya_en_ne = himalaya_en_ne.rename(
    columns={
        "human": "source",
        "gpt": "target"
    }
)
himalaya_en_ne["direction"] = "eng_to_nep"
himalaya_en_ne["dataset"] = "himalaya_eng_nep"


#### Combining the three standardized datasets


In [19]:
# Combine all standardized datasets into one DataFrame.
combined_df = pd.concat(
    [
        iam_df,
        himalaya_ne_en,
        himalaya_en_ne
    ],
    ignore_index=True
)

In [20]:
print("\nFinal shape:", combined_df.shape)

print("\nDataset distribution:")
print(combined_df["dataset"].value_counts())

print("\nDirection distribution:")
print(combined_df["direction"].value_counts())

display(combined_df.head(10))


Final shape: (1099921, 4)

Dataset distribution:
dataset
iamTangsang         724430
himalaya_nep_eng    198159
himalaya_eng_nep    177332
Name: count, dtype: int64

Direction distribution:
direction
nep_to_eng    922589
eng_to_nep    177332
Name: count, dtype: int64


,source,target,direction,dataset
0,"""कुनै पनि अन्य सरकारी एजेन्सीले यो जानकारी प्र...","""No other government agency can use this infor...",nep_to_eng,iamTangsang
1,"एउटा गीत प्ले गर्नुहोस्, जुन तपाईं चाहानुहुन्छ।",Pick a song which you want.,nep_to_eng,iamTangsang
2,"तर, तपाईं उदास हुँदा चिढिएर अझ बढी रिसाउनुहुन्...","But, when you get sad, if you are going to get...",nep_to_eng,iamTangsang
3,जिन्दगी भनेको कुनै चलचित्र होइन ।,"""Life is not a movie.",nep_to_eng,iamTangsang
4,"त्यसपछि, म र उनीबीच केही दूरीको महसुस गरेँ।",Yet I felt there was a distance between me and...,nep_to_eng,iamTangsang
5,बोन्डाउकाउ,Bondoukou,nep_to_eng,iamTangsang
6,यस सन्दर्भमा यहाँ के भन्नुहुन्छ ?,What do you refer to in this case?,nep_to_eng,iamTangsang
7,यस दिन धेरै देशहरुमा राष्ट्रिय अवकाशको घोषणा ग...,This day is declared a national holiday in man...,nep_to_eng,iamTangsang
8,"तर, उनले काम गर्न थालेको...",but he was working...,nep_to_eng,iamTangsang
9,तपाईँको सूचना परिवर्तन गर्नका लागि आफ्नो पासवर...,You must enter your password in order to chang...,nep_to_eng,iamTangsang


In [21]:
# Check for missing values before text cleaning.
print("Missing values:")
print(combined_df[["source", "target", "direction"]].isna().sum())


Missing values:
source       0
target       0
direction    0
dtype: int64


In [22]:
# Convert text to strings, trim surrounding whitespace, and check empty pairs.
combined_df["source"] = combined_df["source"].astype(str).str.strip()
combined_df["target"] = combined_df["target"].astype(str).str.strip()

removed_pairs = combined_df[
    (combined_df["source"] == "") |
    (combined_df["target"] == "")
].copy()
print("Removed pairs:", len(removed_pairs))
display(removed_pairs)

Removed pairs: 10


,source,target,direction,dataset
16116,,While the number of lower secondary schools wa...,nep_to_eng,iamTangsang
108756,,4. Mule tracks:,nep_to_eng,iamTangsang
136931,,It is necessary to recognize that the farmers ...,nep_to_eng,iamTangsang
179190,,Cement: It is estimated that in order to under...,nep_to_eng,iamTangsang
216297,,B.,nep_to_eng,iamTangsang
389281,,The increase in the internal dema nd for agric...,nep_to_eng,iamTangsang
519778,,Out of the total loan and investment disbursem...,nep_to_eng,iamTangsang
582191,,,nep_to_eng,iamTangsang
637770,,D. level.,nep_to_eng,iamTangsang
717626,,"Radio Broadcasting , Television Transmission a...",nep_to_eng,iamTangsang


In [23]:
#Remove the exact pairs
combined_df = combined_df.drop(removed_pairs.index)

print("After removing empty pairs:", combined_df.shape)

After removing empty pairs: (1099911, 4)


In [24]:
def normalize_text(text):
    text = re.sub(r"\s+", " ", text) # Collapse repeated whitespace (spaces, tabs, and line breaks) to one space.
    return text.strip() # Remove space from beginning and the end.


combined_df["source"] = combined_df["source"].apply(normalize_text)
combined_df["target"] = combined_df["target"].apply(normalize_text)

In [25]:
# Inspect exact duplicate source-target pairs before removing them.
duplicates = combined_df[
    combined_df.duplicated(
        subset=["source", "target"],
        keep=False
    )
].sort_values(["source", "target"])

print("Duplicate rows:", len(duplicates))
print("Unique duplicate pairs:",
      duplicates[["source", "target"]].drop_duplicates().shape[0])

display(
    duplicates[
        ["source", "target", "direction", "dataset"]
    ]
)

Duplicate rows: 429333
Unique duplicate pairs: 208665


,source,target,direction,dataset
449765,!,!,nep_to_eng,iamTangsang
671011,!,!,nep_to_eng,iamTangsang
1030108,!,!,eng_to_nep,himalaya_eng_nep
519910,! (# 033;) बिस्मयादिबोधक चिन्ह,! (#033;) Exclamation mark,nep_to_eng,iamTangsang
579763,! (# 033;) बिस्मयादिबोधक चिन्ह,! (#033;) Exclamation mark,nep_to_eng,iamTangsang
...,...,...,...,...
995192, Amended by First amendment ., पहिलो संशोधनद्धारा संशोधित ।,eng_to_nep,himalaya_eng_nep
1010465,11. Radio Paging Network,११. रेडियो पेजिङ्ग नेटवर्क,eng_to_nep,himalaya_eng_nep
1010515,11. Radio Paging Network,११. रेडियो पेजिङ्ग नेटवर्क,eng_to_nep,himalaya_eng_nep
563258,﻿सिकागोको गुडम्यान नाटकघरको `` द मिसान्थ्रोपी ...,In an Oct. 19 review of `` The Misanthrope '' ...,nep_to_eng,iamTangsang


In [26]:
#duplicate dataset-count analysis
duplicate_pairs = (
    combined_df
    .groupby(["source", "target"])["dataset"]
    .nunique()
)

cross_dataset_pairs = duplicate_pairs[
    duplicate_pairs > 1
]

print(
    "Pairs appearing in multiple datasets:",
    len(cross_dataset_pairs)
)


Pairs appearing in multiple datasets: 193076


In [27]:
# Keep one copy of each exact source-target pair.
print("Before:", len(combined_df))

cleaned_df = combined_df.drop_duplicates(
    subset=["source", "target"],
    keep="first"
).reset_index(drop=True)

print("After:", len(cleaned_df))
print("Removed:", len(combined_df) - len(cleaned_df))

Before: 1099911
After: 879243
Removed: 220668


In [28]:
# Check conflicting translations same source but different targets or vice versa
source_target_counts = (
    cleaned_df
    .groupby(["source", "direction"])["target"]
    .nunique()
)

conflicting_sources = source_target_counts[
    source_target_counts > 1
]

print(
    "Sources with multiple translations:",
    len(conflicting_sources)
)

# Same target but different sources
target_source_counts = (
    cleaned_df
    .groupby(["target", "direction"])["source"]
    .nunique()
)

conflicting_targets = target_source_counts[
    target_source_counts > 1
]

print(
    "Targets with multiple sources:",
    len(conflicting_targets)
)

Sources with multiple translations: 119448
Targets with multiple sources: 34542


In [29]:
# inspect the sources with different targets
conflicting_sources_df = (
    cleaned_df[
        cleaned_df["source"].isin(
            conflicting_sources.index.get_level_values("source")
        )
    ]
    .sort_values(["source", "direction"])
)
print("Sources with multiple translations:", len(conflicting_sources_df))
display(
    conflicting_sources_df[
        ["source", "target", "direction", "dataset"]
    ].head(100)
)

Sources with multiple translations: 371715


,source,target,direction,dataset
152552,! -- तपाईँ माथि हुनुहुन्छ ।,! – You’re up.,nep_to_eng,iamTangsang
222320,! -- तपाईँ माथि हुनुहुन्छ ।,! -- You're up.,nep_to_eng,iamTangsang
3703,""" ""उनीहरूले मलाई तपाईं बाहिर जान सक्नुहुन्न, ट...","""But I will not abide by what they say, you ca...",nep_to_eng,iamTangsang
383532,""" ""उनीहरूले मलाई तपाईं बाहिर जान सक्नुहुन्न, ट...","""But I will not abide by what they say, you ca...",nep_to_eng,iamTangsang
183621,""" ""जसले ऊर्जाका बारेमा सोच्दैन, उसले भविष्यका ...",He who does not think of energy is not thinkin...,nep_to_eng,iamTangsang
...,...,...,...,...
580053,""" उनीहरूसँग हतियार भएको आशङ्का छ र उनीहरूलाई ख...","""They are considered armed and dangerous.",nep_to_eng,iamTangsang
608760,""" उनीहरूसँग हतियार भएको आशङ्का छ र उनीहरूलाई ख...","""They are armed and dangerous.",nep_to_eng,iamTangsang
23079,""" एउटा उदाहरण दिनुहोस् जहाँ यो मामला हुन सक्छ।",Please give an example where this could be the...,nep_to_eng,iamTangsang
180719,""" एउटा उदाहरण दिनुहोस् जहाँ यो मामला हुन सक्छ।",Provide an example in which this could happen.,nep_to_eng,iamTangsang


In [30]:
#Normalize ONLY the conflicting targets for comparison
import unicodedata

def normalize_conflicts(text):
  text = text.lower() # Lowercase

  # Remove Unicode punctuation
  text = "".join(
        char for char in text
        if not unicodedata.category(char).startswith("P")
    )
  # Normalize whitespace
  text = re.sub(r"\s+", " ", text)
  return text.strip()

conflicting_sources_df["target_normalized"] = ( conflicting_sources_df["target"] .apply(normalize_conflicts) )


In [31]:
# Display the original target alongside the normalized target
display( conflicting_sources_df[ [ "source", "target", "target_normalized", "direction", "dataset" ] ].head(100) )

,source,target,target_normalized,direction,dataset
152552,! -- तपाईँ माथि हुनुहुन्छ ।,! – You’re up.,youre up,nep_to_eng,iamTangsang
222320,! -- तपाईँ माथि हुनुहुन्छ ।,! -- You're up.,youre up,nep_to_eng,iamTangsang
3703,""" ""उनीहरूले मलाई तपाईं बाहिर जान सक्नुहुन्न, ट...","""But I will not abide by what they say, you ca...",but i will not abide by what they say you cann...,nep_to_eng,iamTangsang
383532,""" ""उनीहरूले मलाई तपाईं बाहिर जान सक्नुहुन्न, ट...","""But I will not abide by what they say, you ca...",but i will not abide by what they say you cann...,nep_to_eng,iamTangsang
183621,""" ""जसले ऊर्जाका बारेमा सोच्दैन, उसले भविष्यका ...",He who does not think of energy is not thinkin...,he who does not think of energy is not thinkin...,nep_to_eng,iamTangsang
...,...,...,...,...,...
580053,""" उनीहरूसँग हतियार भएको आशङ्का छ र उनीहरूलाई ख...","""They are considered armed and dangerous.",they are considered armed and dangerous,nep_to_eng,iamTangsang
608760,""" उनीहरूसँग हतियार भएको आशङ्का छ र उनीहरूलाई ख...","""They are armed and dangerous.",they are armed and dangerous,nep_to_eng,iamTangsang
23079,""" एउटा उदाहरण दिनुहोस् जहाँ यो मामला हुन सक्छ।",Please give an example where this could be the...,please give an example where this could be the...,nep_to_eng,iamTangsang
180719,""" एउटा उदाहरण दिनुहोस् जहाँ यो मामला हुन सक्छ।",Provide an example in which this could happen.,provide an example in which this could happen,nep_to_eng,iamTangsang


In [32]:

# Find rows where the same source has targets that
# become identical after normalization

normalized_duplicates = (
    conflicting_sources_df[
        conflicting_sources_df.duplicated(
            subset=["source", "target_normalized"],
            keep=False
        )
    ]
    .sort_values(["source", "target_normalized"])
)

print("Normalized duplicate rows:", len(normalized_duplicates))

display(
    normalized_duplicates[
        [
            "source",
            "target",
            "target_normalized",
            "direction",
            "dataset"
        ]
    ].head(100)
)



Normalized duplicate rows: 40855


,source,target,target_normalized,direction,dataset
152552,! -- तपाईँ माथि हुनुहुन्छ ।,! – You’re up.,youre up,nep_to_eng,iamTangsang
222320,! -- तपाईँ माथि हुनुहुन्छ ।,! -- You're up.,youre up,nep_to_eng,iamTangsang
183621,""" ""जसले ऊर्जाका बारेमा सोच्दैन, उसले भविष्यका ...",He who does not think of energy is not thinkin...,he who does not think of energy is not thinkin...,nep_to_eng,iamTangsang
199517,""" ""जसले ऊर्जाका बारेमा सोच्दैन, उसले भविष्यका ...",'He who does not think of energy is not thinki...,he who does not think of energy is not thinkin...,nep_to_eng,iamTangsang
392001,""" ""जसले ऊर्जाका बारेमा सोच्दैन, उसले भविष्यका ...","""He who does not think of energy is not thinki...",he who does not think of energy is not thinkin...,nep_to_eng,iamTangsang
...,...,...,...,...,...
292838,"""अब हुनसक्छ कि तपाईंले मेरो कथा पढ्नुभएको छ, त...","'Maybe now that you've read my story, you'll s...",maybe now that youve read my story youll see m...,nep_to_eng,iamTangsang
465247,"""अब हुनसक्छ कि तपाईंले मेरो कथा पढ्नुभएको छ, त...","""Maybe now that you've read my story, you'll s...",maybe now that youve read my story youll see m...,nep_to_eng,iamTangsang
611907,"""अबदेखि क्लबको कडा नेतृत्व हुनेछ।",From now on the club will have strong leadership.,from now on the club will have strong leadership,nep_to_eng,iamTangsang
630100,"""अबदेखि क्लबको कडा नेतृत्व हुनेछ।","""From now on the club will have strong leaders...",from now on the club will have strong leadership,nep_to_eng,iamTangsang


In [33]:
# Keep the first row for each normalized duplicate pair
rows_to_remove = normalized_duplicates.duplicated(
    subset=["source", "target_normalized"],
    keep="first"
)

# Get the indexes of duplicate rows that should be removed
duplicate_indices = normalized_duplicates.index[rows_to_remove]
display(duplicate_indices)
print("Normalized source-target duplicates to remove:", len(duplicate_indices))

# Inspect rows before removing
display(
    cleaned_df.loc[
        duplicate_indices,
        ["source", "target", "direction", "dataset"]
    ].head(50)
)
# Remove those rows from the original cleaned dataset
cleaned_df = cleaned_df.drop(index=duplicate_indices).copy()

print("After removing normalized duplicates:", cleaned_df.shape)


Index([222320, 199517, 392001, 679895, 555782, 876247, 394303, 188011, 593158,
       491819,
       ...
       760970, 802657, 802782, 802798, 802888, 802902, 681582, 182672, 613576,
       456561],
      dtype='int64', length=22426)

Normalized source-target duplicates to remove: 22426


,source,target,direction,dataset
222320,! -- तपाईँ माथि हुनुहुन्छ ।,! -- You're up.,nep_to_eng,iamTangsang
199517,""" ""जसले ऊर्जाका बारेमा सोच्दैन, उसले भविष्यका ...",'He who does not think of energy is not thinki...,nep_to_eng,iamTangsang
392001,""" ""जसले ऊर्जाका बारेमा सोच्दैन, उसले भविष्यका ...","""He who does not think of energy is not thinki...",nep_to_eng,iamTangsang
679895,""" ""जसले ऊर्जाका बारेमा सोच्दैन, उसले भविष्यका ...","""He who does not think of energy is not thinki...",nep_to_eng,iamTangsang
555782,""" ""प्रिय एडिसन महोदय, म दुई महान् मानिसहरूबारे...","My Dear Edison, I know two great men and you a...",nep_to_eng,iamTangsang
876247,""" we struggled to be taken seriously,"" she exp...",""" हामीलाई गम्भीर रुपले लिइयोस भनेर संघर्ष गर्य...",eng_to_nep,himalaya_eng_nep
394303,""" अब यो विचार गर्न सम्भव छैन कि एउटा पक्षले अर...","""It is no longer possible to think that one si...",nep_to_eng,iamTangsang
188011,""" उनले आफ्नो सम्बोधनमा भने, ""आज हामीले सात वर्...","""Today, we follow the will of the people in th...",nep_to_eng,iamTangsang
593158,""" उनले आफ्नो सम्बोधनमा भने, ""आज हामीले सात वर्...","""Today we follow the will of the people in the...",nep_to_eng,iamTangsang
491819,""" उनले आफ्नो सम्बोधनमा भने, ""आज हामीले सात वर्...","""Today, we follow the will of the people in th...",nep_to_eng,iamTangsang


After removing normalized duplicates: (856817, 4)


In [34]:
#inspect the targets with different sources
conflicting_target_df = (
    cleaned_df[
        cleaned_df["target"].isin(
            conflicting_targets.index.get_level_values("target")
        )
    ]
    .sort_values(["target", "direction"])
)
print("Targets with multiple sources:", len(conflicting_target_df))
display(
    conflicting_target_df[
        ["source", "target", "direction", "dataset"]
    ].head(100)
)

Targets with multiple sources: 76411


,source,target,direction,dataset
819890,! -- You're up.,! -- तपाईँ माथि हुनुहुन्छ ।,eng_to_nep,himalaya_eng_nep
843475,! – You’re up.,! -- तपाईँ माथि हुनुहुन्छ ।,eng_to_nep,himalaya_eng_nep
195562,'मैले मेरी छोरीलाई सोधें ।,""" I asked my daughter.",nep_to_eng,iamTangsang
238107,"', मैले छोरीलाई सोधेँ।",""" I asked my daughter.",nep_to_eng,iamTangsang
186461,"""विश्व एउटा पुस्तक हो अनि त्यसको विचरण नगर्नेह...",""" The world is a book and those who don't trav...",nep_to_eng,iamTangsang
...,...,...,...,...
227453,"""तर यो कसरी सम्भव भयो बाबु ?","""Daddy, how can this be?",nep_to_eng,iamTangsang
424811,"""प्रतिमा, यो कसरी हुन सक्छ ?","""Daddy, how can this be?",nep_to_eng,iamTangsang
403250,लोकतन्त्रले विजय हासिल गरेको छ ।,"""Democracy has won.",nep_to_eng,iamTangsang
430959,लोकतन्त्रको जीत भयो ।,"""Democracy has won.",nep_to_eng,iamTangsang


In [35]:
#apply normalization in sources of same target but multiple sources
conflicting_target_df["source_normalized"] = ( conflicting_target_df["source"] .apply(normalize_conflicts) )

In [36]:
display( conflicting_target_df[ [ "source", "target", "source_normalized", "direction", "dataset" ] ].head(100) )

,source,target,source_normalized,direction,dataset
819890,! -- You're up.,! -- तपाईँ माथि हुनुहुन्छ ।,youre up,eng_to_nep,himalaya_eng_nep
843475,! – You’re up.,! -- तपाईँ माथि हुनुहुन्छ ।,youre up,eng_to_nep,himalaya_eng_nep
195562,'मैले मेरी छोरीलाई सोधें ।,""" I asked my daughter.",मैले मेरी छोरीलाई सोधें,nep_to_eng,iamTangsang
238107,"', मैले छोरीलाई सोधेँ।",""" I asked my daughter.",मैले छोरीलाई सोधेँ,nep_to_eng,iamTangsang
186461,"""विश्व एउटा पुस्तक हो अनि त्यसको विचरण नगर्नेह...",""" The world is a book and those who don't trav...",विश्व एउटा पुस्तक हो अनि त्यसको विचरण नगर्नेहर...,nep_to_eng,iamTangsang
...,...,...,...,...,...
227453,"""तर यो कसरी सम्भव भयो बाबु ?","""Daddy, how can this be?",तर यो कसरी सम्भव भयो बाबु,nep_to_eng,iamTangsang
424811,"""प्रतिमा, यो कसरी हुन सक्छ ?","""Daddy, how can this be?",प्रतिमा यो कसरी हुन सक्छ,nep_to_eng,iamTangsang
403250,लोकतन्त्रले विजय हासिल गरेको छ ।,"""Democracy has won.",लोकतन्त्रले विजय हासिल गरेको छ,nep_to_eng,iamTangsang
430959,लोकतन्त्रको जीत भयो ।,"""Democracy has won.",लोकतन्त्रको जीत भयो,nep_to_eng,iamTangsang


In [37]:

# Find rows where the same targets has source that
# become identical after normalization

normalized_duplicates_ts = (
    conflicting_target_df[
        conflicting_target_df.duplicated(
            subset=["source_normalized", "target"],
            keep=False
        )
    ]
    .sort_values(["source_normalized", "target"])
)

print("Normalized duplicate rows:", len(normalized_duplicates_ts))

display(
    normalized_duplicates_ts[
        [
            "source",
            "source_normalized",
            "target",
            "direction",
            "dataset"
        ]
    ].head(100)
)



Normalized duplicate rows: 17076


,source,source_normalized,target,direction,dataset
803374,…,,(क),eng_to_nep,himalaya_eng_nep
803405,……,,(क),eng_to_nep,himalaya_eng_nep
803375,…,,(ख),eng_to_nep,himalaya_eng_nep
803406,……,,(ख),eng_to_nep,himalaya_eng_nep
803376,…,,(ग),eng_to_nep,himalaya_eng_nep
...,...,...,...,...,...
862341,A disk flipping game derived from Reversi. Iag...,a disk flipping game derived from reversi iagn...,रिभर्सीबाट प्राप्त खेल फ्लिपिङ गर्ने डिस्क । इ...,eng_to_nep,himalaya_eng_nep
871782,A disk flipping game derived from Reversi Iagn...,a disk flipping game derived from reversi iagn...,रिभर्सीबाट प्राप्त खेल फ्लिपिङ गर्ने डिस्क । इ...,eng_to_nep,himalaya_eng_nep
811892,A House aide suggested that Mr. Phelan was so ...,a house aide suggested that mr phelan was so v...,"संसदका एक सहायकले श्री फेलान यति ""अस्पष्ट र नर...",eng_to_nep,himalaya_eng_nep
811907,A House aide suggested that Mr. Phelan was so ...,a house aide suggested that mr phelan was so v...,"संसदका एक सहायकले श्री फेलान यति ""अस्पष्ट र नर...",eng_to_nep,himalaya_eng_nep


In [38]:
rows_to_remove_ts = normalized_duplicates_ts.duplicated(
    subset=["target", "source_normalized"],
    keep="first"
)

duplicate_indices_ts = normalized_duplicates_ts.index[rows_to_remove_ts]

print("Normalized target-source duplicates to remove:", len(duplicate_indices_ts))
display(duplicate_indices_ts)
# Inspect rows before removing
display(
    cleaned_df.loc[
        duplicate_indices_ts,
        ["source", "target", "direction", "dataset"]
    ].head(100)
)

# Remove duplicates
cleaned_df = cleaned_df.drop(index=duplicate_indices_ts).copy()

print("After removing normalized target-source duplicates:", cleaned_df.shape)



Normalized target-source duplicates to remove: 8693


Index([803405, 803406, 803407, 386186, 787296, 797112, 788013, 788055, 797111,
       789143,
       ...
       549353, 697796,  90138, 677580, 593312, 189893, 376396, 491694, 540769,
        95247],
      dtype='int64', length=8693)

,source,target,direction,dataset
803405,……,(क),eng_to_nep,himalaya_eng_nep
803406,……,(ख),eng_to_nep,himalaya_eng_nep
803407,……,(ग),eng_to_nep,himalaya_eng_nep
386186,।,.,nep_to_eng,iamTangsang
787296,......................................,.......................,eng_to_nep,himalaya_eng_nep
...,...,...,...,...
873040,Apply filters to new messages in INBOX,प्राप्तमञ्जूषा भित्र नयाँ सन्देशहरूमा फिल्टर ल...,eng_to_nep,himalaya_eng_nep
820415,Apply Layer Mask,तह मास्क लागू गर्नुहोस्,eng_to_nep,himalaya_eng_nep
863812,"Are you sure you want to permanently delete ""%B""?",तपाईँ लाई स्थायीरूपमा मेट्न निश्चित हुनुहुन्छ?,eng_to_nep,himalaya_eng_nep
840177,"Are you sure you want to permanently delete ""%...",तपाईँ रद्दीटोकरीबाट स्थायीरूपमा मेट्न निश्चित ...,eng_to_nep,himalaya_eng_nep


After removing normalized target-source duplicates: (848124, 4)


In [39]:
# Re-check dataset and direction distributions after deduplication.
print(cleaned_df["dataset"].value_counts())
print(cleaned_df["direction"].value_counts())

dataset
iamTangsang         676644
himalaya_eng_nep    171480
Name: count, dtype: int64
direction
nep_to_eng    676644
eng_to_nep    171480
Name: count, dtype: int64


#### Language/Script validation

In [40]:
import re

def has_devanagari(text):
    return bool(re.search(r'[\u0900-\u097F]', text))

def has_latin(text):
    return bool(re.search(r'[A-Za-z]', text))


# Flag rows where either side contains the wrong script
mixed_script = cleaned_df[
    (
        # Nepali → English:
        # source should NOT contain Latin
        # target should NOT contain Devanagari
        (cleaned_df["direction"] == "nep_to_eng") &
        (
            cleaned_df["source"].apply(has_latin) |
            cleaned_df["target"].apply(has_devanagari)
        )
    )
    |
    (
        # English → Nepali:
        # source should NOT contain Devanagari
        # target should NOT contain Latin
        (cleaned_df["direction"] == "eng_to_nep") &
        (
            cleaned_df["source"].apply(has_devanagari) |
            cleaned_df["target"].apply(has_latin)
        )
    )
]

print("Potentially mixed-script rows:", len(mixed_script))

display(
    mixed_script[
        ["source", "target", "direction", "dataset"]
    ]
)

Potentially mixed-script rows: 11087


,source,target,direction,dataset
46368,तपाईंलाई थाहा छ कि तपाईं सक्नुहुन्छ सम्म बचत ग...,Do you know that you can इस हद तक बचाएं 90% on...,nep_to_eng,iamTangsang
83671,"तथापि, तपाईंलाई यो जान्नु आवश्यक छ कि यस प्रस्...","हालाँकि, you need to know that the winnings fr...",nep_to_eng,iamTangsang
108695,": हेर लागि, पृथ्वीको राजा भेला गरिएको छ; तिनीह...","47:5 कारण, the kings of the earth have been ga...",nep_to_eng,iamTangsang
114953,": र यसरी, म सियोनमा मा स्थापित भएको थियो.","24:15 और इस तरह, I was established in Zion .",nep_to_eng,iamTangsang
123550,रेडियो र्टचलाइटमा प्रयोग गरिसकेपछि फालेको पुरा...,"After using the radio urchchलाइट, charging fro...",nep_to_eng,iamTangsang
...,...,...,...,...
879090,This post by Todd Reubold was originally publi...,टड रिउबोल्डद्वारा लिखित यो आलेख पहिला Ensia.co...,eng_to_nep,himalaya_eng_nep
879127,"""#EgyptAir hijacker has explosives…. How did h...","""#EgyptAir इजिप्टएअर अपरहणकर्तासँग विष्फोटक पद...",eng_to_nep,himalaya_eng_nep
879181,License: CC BY-NC-ND 2.0 (No derivatives).,लाइसेंस CC BY - NC - 2.0 ( वास्तविक ),eng_to_nep,himalaya_eng_nep
879186,#Nepal pic.twitter.com/3GCxe07lwV — Nepal In P...,गाजा र चरेस येहा पाइन्छ | Marijuana was legal ...,eng_to_nep,himalaya_eng_nep


In [41]:
# Word counts for source and target
cleaned_df["source_word_count"] = cleaned_df["source"].str.split().str.len()
cleaned_df["target_word_count"] = cleaned_df["target"].str.split().str.len()

# Symmetric length ratio:
# always = larger side / smaller side
cleaned_df["length_ratio"] = (
    cleaned_df[["source_word_count", "target_word_count"]].max(axis=1)
    /
    cleaned_df[["source_word_count", "target_word_count"]].min(axis=1)
)



In [42]:
# Inspect how many examples would be affected by increasingly large length ratios.
for threshold in [5, 10, 20, 50, 100]:
    count = (cleaned_df["length_ratio"] >= threshold).sum()
    print(f"Ratio >= {threshold}: {count} rows")

Ratio >= 5: 868 rows
Ratio >= 10: 266 rows
Ratio >= 20: 87 rows
Ratio >= 50: 11 rows
Ratio >= 100: 3 rows


In [43]:
extreme_5 = (
    cleaned_df[
        cleaned_df["length_ratio"] >= 5
    ]
    .sort_values("length_ratio", ascending=False)
)

print("Total pairs with ratio >= 5:", len(extreme_5))

display(
    extreme_5[
        [
            "source",
            "target",
            "source_word_count",
            "target_word_count",
            "length_ratio",
            "direction",
            "dataset"
        ]
    ]
)

Total pairs with ratio >= 5: 868


,source,target,source_word_count,target_word_count,length_ratio,direction,dataset
777854,Legal defence: In order to make a petition to ...,ज्ञण्.,142,1,142.0,eng_to_nep,himalaya_eng_nep
781275,In the case of temporary teachers no age bar s...,..............,117,1,117.0,eng_to_nep,himalaya_eng_nep
790785,(2) Upon examination/inquiry of the complain r...,६.,114,1,114.0,eng_to_nep,himalaya_eng_nep
798556,"While submitting an application thus, the citi...",८.,87,1,87.0,eng_to_nep,himalaya_eng_nep
796457,The landowner who has the land upto upper limi...,...................,86,1,86.0,eng_to_nep,himalaya_eng_nep
...,...,...,...,...,...,...,...
78438,विविध (क) परिवार नियोजन,"To provide a part of this training, a medical ...",4,20,5.0,nep_to_eng,iamTangsang
44502,शीर्षकनमुनापोर्टमार्ग:,Title: Model: Port: Path: UUID:,1,5,5.0,nep_to_eng,iamTangsang
809567,: -,हालको वडा नं. ः– टोल ः– साविक वडा नं. ः–,2,10,5.0,eng_to_nep,himalaya_eng_nep
809581,138........................,समिति आफैले मिलाउनु पर्नेछ ।,1,5,5.0,eng_to_nep,himalaya_eng_nep


dropping all length ratio> 5

In [44]:
# Remove pairs with a source/target word-count ratio of 5 or greater.
cleaned_df = cleaned_df[
    cleaned_df["length_ratio"] < 5
].copy()

print("Remaining rows:", len(cleaned_df))

Remaining rows: 847256


In [45]:
# Find pairs where the source and target are exactly identical.
same_text = cleaned_df[
    cleaned_df["source"] == cleaned_df["target"]
]
print("Same source and target:", len(same_text))
same_text[["source", "target", "dataset"]]

Same source and target: 104


,source,target,dataset
186,????????,????????,iamTangsang
2096,⅗,⅗,iamTangsang
15404,??,??,iamTangsang
16157,−,−,iamTangsang
21717,°,°,iamTangsang
...,...,...,...
876161,CC: BY -SA 4..0,CC: BY -SA 4..0,himalaya_eng_nep
876256,Fonte: eritrea-chat.com,Fonte: eritrea-chat.com,himalaya_eng_nep
876983,EU/ECHO/Martin Karimi.,EU/ECHO/Martin Karimi.,himalaya_eng_nep
877715,Photo by Flickr user Trocaire. CC BY 2.0,Photo by Flickr user Trocaire. CC BY 2.0,himalaya_eng_nep


In [46]:
# Remove identical source-target pairs because they are not useful translation examples.
cleaned_df = cleaned_df.drop(same_text.index).copy()

In [47]:
same_text_check = cleaned_df[
    cleaned_df["source"] == cleaned_df["target"]
]

print("Same source and target remaining:", len(same_text_check))

Same source and target remaining: 0


In [48]:
# Create a reversed copy so every valid pair can also be used in the opposite direction.
def reverse_translation(row):
    if row["direction"] == "nep_to_eng":
        new_direction = "eng_to_nep"
    else:
        new_direction = "nep_to_eng"

    return pd.Series({
        "source": row["target"],
        "target": row["source"],
        "direction": new_direction,
        "dataset": row["dataset"]
    })


reversed_df = cleaned_df.apply(reverse_translation, axis=1)

In [49]:
# Add the reversed translation pairs to the original combined dataset.
cleaned_df = pd.concat(
    [
        cleaned_df,
        reversed_df
    ],
    ignore_index=True
)

In [50]:
# Confirm the new total size and direction balance after reversal.
print("Original + reversed:", len(cleaned_df))

print(cleaned_df["direction"].value_counts())

Original + reversed: 1694304
direction
nep_to_eng    847152
eng_to_nep    847152
Name: count, dtype: int64


In [51]:
# Inspect ALL duplicate translation pairs
duplicates = cleaned_df[
    cleaned_df.duplicated(
        subset=["source", "target"],
        keep=False
    )
].sort_values(["source", "target"])

print("Duplicate rows:", len(duplicates))
print("Unique duplicate pairs:",
      duplicates[["source", "target"]].drop_duplicates().shape[0])

display(
    duplicates[
        ["source", "target", "direction", "dataset"]
    ]
)

Duplicate rows: 348116
Unique duplicate pairs: 174058


,source,target,direction,dataset
845919,! !,जुली गूडिसनले सुझाइन्ः,eng_to_nep,himalaya_eng_nep
1270067,! !,जुली गूडिसनले सुझाइन्ः,eng_to_nep,iamTangsang
494394,! (# 033;) बिस्मयादिबोधक चिन्ह,! (#033;) Exclamation mark,nep_to_eng,iamTangsang
1680094,! (# 033;) बिस्मयादिबोधक चिन्ह,! (#033;) Exclamation mark,nep_to_eng,himalaya_eng_nep
832942,! (#033;) Exclamation mark,! (# 033;) बिस्मयादिबोधक चिन्ह,eng_to_nep,himalaya_eng_nep
...,...,...,...,...
1686824,₫(# 8363;) डङ चिन्ह,₫(#8363;) Dong sign,nep_to_eng,himalaya_eng_nep
839672,₫(#8363;) Dong sign,₫(# 8363;) डङ चिन्ह,eng_to_nep,himalaya_eng_nep
936442,₫(#8363;) Dong sign,₫(# 8363;) डङ चिन्ह,eng_to_nep,iamTangsang
846479,﻿The price of not caring,बेपरवाहको परिणाम,eng_to_nep,himalaya_eng_nep


In [52]:
print("Before:", len(cleaned_df))

cleaned1_df = cleaned_df.drop_duplicates(
    subset=["source", "target"],
    keep="first"
).reset_index(drop=True)

print("After:", len(cleaned1_df))
print("Removed:", len(cleaned_df) - len(cleaned1_df))

Before: 1694304
After: 1520246
Removed: 174058


In [53]:
print(cleaned1_df["dataset"].value_counts())
print(cleaned1_df["direction"].value_counts())

dataset
iamTangsang         1265855
himalaya_eng_nep     254391
Name: count, dtype: int64
direction
nep_to_eng    760123
eng_to_nep    760123
Name: count, dtype: int64


In [54]:
print(len(cleaned1_df))

1520246


In [55]:
cleaned1_df[["source", "target", "dataset", "direction"]]

,source,target,dataset,direction
0,"""कुनै पनि अन्य सरकारी एजेन्सीले यो जानकारी प्र...","""No other government agency can use this infor...",iamTangsang,nep_to_eng
1,"एउटा गीत प्ले गर्नुहोस्, जुन तपाईं चाहानुहुन्छ।",Pick a song which you want.,iamTangsang,nep_to_eng
2,"तर, तपाईं उदास हुँदा चिढिएर अझ बढी रिसाउनुहुन्...","But, when you get sad, if you are going to get...",iamTangsang,nep_to_eng
3,जिन्दगी भनेको कुनै चलचित्र होइन ।,"""Life is not a movie.",iamTangsang,nep_to_eng
4,"त्यसपछि, म र उनीबीच केही दूरीको महसुस गरेँ।",Yet I felt there was a distance between me and...,iamTangsang,nep_to_eng
...,...,...,...,...
1520241,लाइसेंस CC BY - NC - 2.0 ( वास्तविक ),License: CC BY-NC-ND 2.0 (No derivatives).,himalaya_eng_nep,nep_to_eng
1520242,गाजा र चरेस येहा पाइन्छ | Marijuana was legal ...,#Nepal pic.twitter.com/3GCxe07lwV — Nepal In P...,himalaya_eng_nep,nep_to_eng
1520243,फाइन्डिङ्ग निमो (निमोलाई खोज्दा),Photo: Abstract Stories / Vkontakte,himalaya_eng_nep,nep_to_eng
1520244,फ्लिकर प्रयोगकर्ता निव सिङ्गरको तस्वीर (CC BY-...,Photo by Flickr user Niv Singer. (CC BY-SA).,himalaya_eng_nep,nep_to_eng


In [56]:
final_df = cleaned1_df[
    ["source", "target", "direction"]
].copy()
print("Final dataset shape:", final_df.shape)

print("\nDirection distribution:")
print(final_df["direction"].value_counts())

display(final_df.head(10))

Final dataset shape: (1520246, 3)

Direction distribution:
direction
nep_to_eng    760123
eng_to_nep    760123
Name: count, dtype: int64


,source,target,direction
0,"""कुनै पनि अन्य सरकारी एजेन्सीले यो जानकारी प्र...","""No other government agency can use this infor...",nep_to_eng
1,"एउटा गीत प्ले गर्नुहोस्, जुन तपाईं चाहानुहुन्छ।",Pick a song which you want.,nep_to_eng
2,"तर, तपाईं उदास हुँदा चिढिएर अझ बढी रिसाउनुहुन्...","But, when you get sad, if you are going to get...",nep_to_eng
3,जिन्दगी भनेको कुनै चलचित्र होइन ।,"""Life is not a movie.",nep_to_eng
4,"त्यसपछि, म र उनीबीच केही दूरीको महसुस गरेँ।",Yet I felt there was a distance between me and...,nep_to_eng
5,बोन्डाउकाउ,Bondoukou,nep_to_eng
6,यस सन्दर्भमा यहाँ के भन्नुहुन्छ ?,What do you refer to in this case?,nep_to_eng
7,यस दिन धेरै देशहरुमा राष्ट्रिय अवकाशको घोषणा ग...,This day is declared a national holiday in man...,nep_to_eng
8,"तर, उनले काम गर्न थालेको...",but he was working...,nep_to_eng
9,तपाईँको सूचना परिवर्तन गर्नका लागि आफ्नो पासवर...,You must enter your password in order to chang...,nep_to_eng


In [57]:
# Start with the final cleaned dataframe
df = final_df.copy()

# Shuffle rows reproducibly
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [58]:
# Create group key so both translation directions stay together
def make_group_key(row):
    return tuple(sorted([row["source"], row["target"]]))

df["group_key"] = df.apply(make_group_key, axis=1)

print("Total rows:", len(df))
print("Unique groups:", df["group_key"].nunique())

Total rows: 1520246
Unique groups: 760123


In [60]:
import numpy as np
# Validation target
TARGET_VAL_ROWS = 10000
SEED = 42

# Number of rows in each translation group
group_sizes = df.groupby("group_key").size()

# Get all unique groups
unique_groups = group_sizes.index.to_numpy()

# Randomize the groups
rng = np.random.default_rng(SEED)
shuffled_groups = rng.permutation(unique_groups)

# Select whole groups for validation
val_groups = set()
running_total = 0

for g in shuffled_groups:
    if running_total >= TARGET_VAL_ROWS:
        break

    val_groups.add(g)
    running_total += group_sizes[g]

print("Selected groups:", len(val_groups))
print("Validation rows:", running_total)

Selected groups: 5000
Validation rows: 10000


In [61]:
# Identify validation rows
is_val = df["group_key"].isin(val_groups)

# Create train and validation DataFrames
val_df = df[is_val].copy()
train_df = df[~is_val].copy()

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))

Train rows: 1510246
Validation rows: 10000


In [62]:
display(val_df.head(20))
print(val_df["direction"].value_counts())

,source,target,direction,group_key
223,पागल नै होइन।,It is not insane.,nep_to_eng,"(It is not insane., पागल नै होइन।)"
417,रोम्बलन,Romblon,nep_to_eng,"(Romblon, रोम्बलन)"
670,"प्रेतहरूलाई नाक, आँखा, कान, टाउकोको शीर्ष भाग ...",Demons can also be expelled out through the no...,nep_to_eng,(Demons can also be expelled out through the n...
851,उनी आफ्नो अवस्थाबाट यति निराश थिए कि १० वर्षको...,He was so frustrated with his disability that ...,nep_to_eng,(He was so frustrated with his disability that...
1071,But many people did not understand.,तर धेरैले यो कुरा बुझेका हुन्नन्।,eng_to_nep,"(But many people did not understand., तर धेरैल..."
1445,कोशि गण्डकीको विनास लिला यो वर्ष मात्र अकस्मात...,"Vinas Lila of Kochi Gandaki, this year has not...",nep_to_eng,"(Vinas Lila of Kochi Gandaki, this year has no..."
3400,तपाईं धेरै पैसा बनाउन हुनेछ।,You will make a lot of money.,nep_to_eng,"(You will make a lot of money., तपाईं धेरै पैस..."
3551,It also reduces competition.,यसले प्रतियोगिताको गरिमा पनि घटाउँछ ।,eng_to_nep,"(It also reduces competition., यसले प्रतियोगित..."
3757,These are the causes!,यस्ता छन् कारणहरू !,eng_to_nep,"(These are the causes!, यस्ता छन् कारणहरू !)"
3801,WEIBULL:probability density or cumulative dist...,R.PWEIBULL:Weibull वितरणको क्युमुलेटिभ वितरण क...,eng_to_nep,(R.PWEIBULL:Weibull वितरणको क्युमुलेटिभ वितरण ...


direction
nep_to_eng    5000
eng_to_nep    5000
Name: count, dtype: int64


In [63]:
# Verify that no translation group exists in both splits
train_groups = set(train_df["group_key"])
val_groups_check = set(val_df["group_key"])

overlap = train_groups & val_groups_check

print("Overlapping groups:", len(overlap))

Overlapping groups: 0


In [64]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

eval_dataset = Dataset.from_pandas(
    val_df,
    preserve_index=False
)

final_split_dataset = DatasetDict({
    "train": train_dataset,
    "validation": eval_dataset
})

In [ ]:
from huggingface_hub import login

login()

In [ ]:
REPO_ID = "sjcMLT/nep-eng-parallel-corpus"
final_split_dataset.push_to_hub(
    REPO_ID,
    private=True)

In [67]:
from datasets import load_dataset

check = load_dataset("sjcMLT/nep-eng-parallel-corpus")
print(check)
print(check["train"][0])
print(check["validation"][0])


README.md:   0%|          | 0.00/476 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  193MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.30MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1510246 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source', 'target', 'direction'],
        num_rows: 1510246
    })
    validation: Dataset({
        features: ['source', 'target', 'direction'],
        num_rows: 10000
    })
})
{'source': 'It looks just as before.', 'target': 'विगत हेर्दा यस्तै लाग्छ ।', 'direction': 'eng_to_nep'}
{'source': 'उनीहरू त्यो विषयमा प्रायः बहस गरिरहन्छन्।', 'target': 'They often argued about this topic.', 'direction': 'nep_to_eng'}
